# 🚀 Train YOLOv8 cho TikTok Auto Bot

Notebook này train YOLOv8 nano để phát hiện nút TikTok.
Dữ liệu được thu thập tự động từ chế độ Record của bot.

## Bước 1: Cài đặt thư viện

In [ ]:
!pip install ultralytics

## Bước 2: Upload dataset

Nén thư mục `dataset/` từ máy tính thành `dataset.zip` rồi upload lên đây.

Hoặc kết nối Google Drive:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Nếu dataset nằm trong Drive:
# !cp -r /content/drive/MyDrive/path/to/dataset /content/dataset

In [ ]:
# Upload dataset.zip từ máy
from google.colab import files
import zipfile
import os

uploaded = files.upload()  # Chọn file dataset.zip

for filename in uploaded.keys():
    if filename.endswith('.zip'):
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('/content/')
        print(f"Đã giải nén {filename}")

os.listdir('/content/dataset')

## Bước 3: Tạo data.yaml

In [ ]:
import os

# Tìm tất cả class (thư mục con trong dataset/)
dataset_dir = '/content/dataset'
classes = sorted([
    d for d in os.listdir(dataset_dir)
    if os.path.isdir(os.path.join(dataset_dir, d))
])

print(f"Classes: {classes}")
print(f"Tổng: {len(classes)} class")

# Đếm số ảnh mỗi class
total_images = 0
for cls in classes:
    count = len([f for f in os.listdir(os.path.join(dataset_dir, cls)) if f.endswith('.png')])
    total_images += count
    print(f"  {cls}: {count} ảnh")

print(f"Tổng: {total_images} ảnh")

if total_images < 50:
    print("⚠️ CẢNH BÁO: Dưới 50 ảnh, nên thu thập thêm để kết quả tốt hơn!")
else:
    print("✅ Đủ dữ liệu để train!")

In [ ]:
# Tạo data.yaml
yaml_content = f"""
path: {dataset_dir}
train: {dataset_dir}
val: {dataset_dir}

nc: {len(classes)}
names: {classes}
"""

yaml_path = '/content/data.yaml'
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(yaml_content)
print(f"Đã tạo {yaml_path}")

## Bước 4: Train YOLOv8 nano

In [ ]:
from ultralytics import YOLO

# Load model nền (pre-trained)
model = YOLO('yolov8n.pt')

# Train
# epochs: số lần lặp (100-200 cho dataset nhỏ)
# imgsz: kích thước ảnh (640 là chuẩn)
# batch: batch size (16 cho GPU T4, giảm nếu hết RAM)
results = model.train(
    data='/content/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    name='tiktok_buttons',
    patience=20,       # Dừng sớm nếu không cải thiện sau 20 epoch
    lr0=0.001,         # Learning rate thấp cho fine-tune
)

## Bước 5: Đánh giá model

In [ ]:
# Validate
metrics = model.val()
print(f"mAP50: {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")

In [ ]:
# Test trên ảnh mẫu
import glob
test_images = []
for cls in classes:
    imgs = glob.glob(f'{dataset_dir}/{cls}/*.png')[:2]
    test_images.extend(imgs)

if test_images:
    results = model(test_images[:4])
    for r in results:
        r.show()

## Bước 6: Tải model về máy

In [ ]:
# Model đã train nằm ở:
# runs/detect/tiktok_buttons/weights/best.pt

from google.colab import files
files.download('runs/detect/tiktok_buttons/weights/best.pt')

print("\n📋 Sau khi tải về:")
print("1. Copy best.pt vào thư mục models/ trong project")
print("2. Khởi động lại bot → YOLO sẽ tự động được kích hoạt")
print("3. Gõ 'yolo' trong chat để kiểm tra trạng thái")

## 🎉 Xong!

Model YOLOv8 đã sẵn sàng để phát hiện nút TikTok.

### Tăng độ chính xác:
- Train thêm epoch (200-300)
- Thu thập thêm ảnh (200+ mỗi class)
- Dùng YOLOv8s (small) thay vì nano
- Data augmentation: thay đổi độ sáng, góc xoay trong config